# step03_suna_bottle_calibration - SUNA Stage 3: bottle-bias correction

The co-located profiles from Stage 2 are geometrically correct but still sit on the **raw SUNA offset** (surface
near -2 to -3 uM instead of ~0). This stage removes that bias by comparing the SUNA to the discrete bottle
nitrate from the rosette and fitting a correction, the standard way a SUNA is brought onto an absolute scale
(Johnson et al., 2018; Zheng et al., 2024).

**The method, and why it is a regression and not a single offset.** A SUNA bias has two parts: a fixed offset
(a baseline error, constant with concentration) and a gain (a slope error, proportional to concentration). So
the honest correction fits both: `SUNA = a + k * bottle`, then inverts it, `nitrate_corrected = (SUNA - a) / k`.
On P45_06 cast 06 the fit was `SUNA = -1.19 + 0.884 * bottle`: an offset of -1.2 uM and a gain of 0.88 (the SUNA
reads about 12% low at high nitrate, consistent with the onboard bromide-temperature compensation being off).
A pure offset (`k` forced to 1) is available via `FIT_SLOPE = False` for comparison.

**How SUNA is matched to a bottle.** Bottle trip times on this cruise are recorded as the cast time, not the
exact firing instant, so matching is done by **depth**, not time: for each bottle (cast, depth) the SUNA value
is the median of the L1-clean co-located frames within `DEPTH_WIN` db of that depth (bottle depth in m is taken
as pressure in db, exact to about 0.5% on the shelf). Using the median over a depth window, and both cast legs,
makes the match robust to the response-time lag seen in Stage 2.

**What is usable, from your pairing audit.** Only casts that have BOTH SUNA and bottles can calibrate: on
P45_06 that is CTD_06 (P3), CTD_08 (J2) and CTD_03 (P1). CTD_03 is flagged (59 min pairing gap) and is included
or excluded with `INCLUDE_FLAGGED`. The deep, high-nitrate points come only from CTD_06, so until CTD_08 and
CTD_03 `.cnv` files are added the **slope rests on a single cast**; the notebook states this in its output.

**Calibration file note (stated explicitly, as requested).** The onboard nitrate was computed with
`SNA2503E.cal`, which is not currently in hand. This stage does NOT re-derive nitrate from the spectra, so it
does not use the cal at all: it corrects the onboard nitrate empirically against the bottles. The `SNA2503C.cal`
in the metadata folder is used only for the wavelength grid in the QC stages. When the `SNA2503E.cal` is
retrieved from the instrument, a physics-based temperature and salinity re-derivation (Sakamoto TCSS) can be
inserted before this step; the bottle correction here already absorbs the mean temperature and salinity bias, so
the corrected product is usable in the meantime.

Sources: Johnson et al. (2018), *JTECH* (BGC-Argo nitrate bias correction against reference samples); Zheng et
al. (2024), *L&O: Methods* (bias-corrected high-resolution SUNA profiles from CTD casts); Sakamoto, Johnson and
Coletti (2009) (temperature and salinity dependence, for the deferred re-derivation).

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib

INTERACTIVE_ZOOM = True
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM = False
    try: get_ipython().run_line_magic("matplotlib", "inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

try:
    _anchor = Path(__file__).resolve()
except NameError:
    _anchor = Path.cwd()
for _p in (_anchor, *_anchor.parents):
    if (_p / "project_paths.py").exists():
        sys.path.insert(0, str(_p)); break
from project_paths import project_root, product_dir
# Text times: suncat.timeutil.to_utc reads ISO8601 first (pandas 2 writes whole and fractional seconds in
# different forms in ONE column; pandas' format guess from the first value then drops or rejects the other form),
# then a per-value pass, and counts anything still unreadable aloud.
try:
    from suncat.timeutil import to_utc, epoch_seconds
except ImportError:                                   # suncat not installed: use the copy in the repo
    sys.path.insert(0, str(Path(_p) / "src"))
    from suncat.timeutil import to_utc, epoch_seconds
from suncat.nitrate import calibration as HC

CRUISE_ID  = "P45_07"
CTD_ROOT   = project_root()
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
L2_FILE    = product_dir(CRUISE_ID, "suna_l2") / f"{CRUISE_ID}__suna_L2_colocated.csv"
BOTTLE_NO3 = CRUISE_DIR / "metadata" / "bottle_nitrate" / f"{CRUISE_ID}_bottle_nitrate.csv"
OUT_DIR    = CRUISE_DIR / "_audit" / "suna"; OUT_DIR.mkdir(parents=True, exist_ok=True)
L3_DIR     = product_dir(CRUISE_ID, "suna_calibrated");     L3_DIR.mkdir(parents=True, exist_ok=True)

# --- bottle-to-SUNA matching -------------------------------------------------
# Registration is not the issue (step02 clocks agree to <0.3 s; time matching sat
# on the right pressure). The remaining disagreement is SUNA flow-cell memory /
# response lag on the upcast bottle stops: in stratified water the cell has not
# flushed, so the SUNA reads deeper, higher-nitrate water than the bottle.
# So calibrate ONLY where the water is vertically homogeneous, identified by the
# SUNA downcast and upcast agreeing at the bottle depth (hysteresis <= HYST_TOL);
# there the SUNA value is trustworthy regardless of lag.
DEPTH_WIN       = 2.5     # db window around the bottle depth, per leg
HYST_TOL        = 2.0     # uM: max |down - up| to call the water homogeneous
MIN_PER_LEG     = 3       # clean SUNA frames needed on EACH leg in the window (was 3 in total, either leg: until
                          # Nov 2026 one frame of one leg could pass a bottle, 5 on P45_06; suncat.nitrate.calibration)
SUNA_VALUE      = "pooled"  # SUNA at a bottle: "pooled" median of the window (P45_07's adopted fit) | "legs" = mean
                          # of the down and up medians (cancels a symmetric lag); the other one is printed as a check
FIT_SLOPE       = True    # True: fit offset+gain; False: offset only (gain=1)
INCLUDE_FLAGGED = False   # include bottles whose 'flag' column is non-empty
print("cruise:", CRUISE_ID, "| L2:", L2_FILE.name, "| bottles:", BOTTLE_NO3.name, "| L3 out:", L3_DIR)

In [ ]:
# ------------------------------------------------------------------ match SUNA to bottles, fit calibration
L2 = pd.read_csv(L2_FILE)
L2["time_utc"] = to_utc(L2["time_utc"], what="L2 co-located").dt.tz_localize(None)   # naive UTC, as written
bot = pd.read_csv(BOTTLE_NO3)
bot.columns = [c.strip() for c in bot.columns]
if "flag" not in bot.columns: bot["flag"] = ""
bot["flag"] = bot["flag"].fillna("")

def suna_legs(cast, depth, win=DEPTH_WIN):
    """SUNA downcast and upcast medians within +/- win db of the bottle depth (suncat.nitrate.calibration)."""
    m = HC.leg_medians(HC._clean(L2), cast, depth, win)
    return dict(down=m["down"], ndn=m["n_down"], up=m["up"], nup=m["n_up"], suna=m["pooled"], n=m["n"])

cal = HC.homogeneous_pairs(L2, bot, DEPTH_WIN, HYST_TOL, MIN_PER_LEG, INCLUDE_FLAGGED, SUNA_VALUE)
fit = cal[cal["used"]]
print(f"{len(fit)} homogeneous bottle-SUNA pairs from {fit['cast'].nunique()} cast(s) "
      f"(of {int((cal['suna'].notna()).sum())} matched; each leg needs {MIN_PER_LEG} clean frames within "
      f"+-{DEPTH_WIN:g} db and |down - up| <= {HYST_TOL:g} uM)\n")
with pd.option_context("display.width",240):
    print(cal.round({"suna_down":2,"suna_up":2,"hyst":2,"suna":3}).to_string(index=False))

# the same fit under the rule before Nov 2026 (3 frames in the window in total) and with the other SUNA value
_cmp = []
for _lab, _kw in ((f"this rule ({MIN_PER_LEG} per leg, {SUNA_VALUE})", dict(min_per_leg=MIN_PER_LEG, value=SUNA_VALUE)),
                  ("rule before Nov 2026 (3 in total)", dict(min_total=3, value=SUNA_VALUE)),
                  (f"{MIN_PER_LEG} per leg, value {'legs' if SUNA_VALUE == 'pooled' else 'pooled'}",
                   dict(min_per_leg=MIN_PER_LEG, value="legs" if SUNA_VALUE == "pooled" else "pooled"))):
    _c = HC.homogeneous_pairs(L2, bot, DEPTH_WIN, HYST_TOL, include_flagged=INCLUDE_FLAGGED, **_kw)
    _f = _c[_c["used"]]; _r = HC.fit_line(_f["bottle"], _f["suna"], FIT_SLOPE)
    _cmp.append((_lab, _r, set(zip(_f["cast"], _f["depth_m"]))))
print("\ncheck: the fit under each rule")
for _lab, _r, _ in _cmp:
    print(f"  {_lab:38s} n {_r['n']:3d}  gain {_r['gain']:.4f}  offset {_r['offset']:+.3f}  R2 {_r['R2']:.4f}")
_gone = sorted(_cmp[1][2] - _cmp[0][2])
if _gone:
    print("  bottles the old rule used and this one does not: " + "; ".join(f"{c} {z:g} m" for c, z in _gone))
# how much the fit rests on single bottles
if len(fit) >= 4:
    _loo = HC.leave_one_out(fit, FIT_SLOPE)
    _k = (_loo["gain"] - HC.fit_line(fit["bottle"], fit["suna"], FIT_SLOPE)["gain"]).abs().idxmax()
    print(f"  leave one out: gain {_loo['gain'].min():.4f} to {_loo['gain'].max():.4f}, offset "
          f"{_loo['offset'].min():+.3f} to {_loo['offset'].max():+.3f}; {int((fit['bottle'] > 2).sum())} of {len(fit)} "
          f"bottles above 2 uM; without {_loo.at[_k, 'cast']} {_loo.at[_k, 'depth_m']:g} m the gain is {_loo.at[_k, 'gain']:.4f}")

OFFSET = GAIN = np.nan; R2 = np.nan
if len(fit) >= 2:
    b = fit["bottle"].to_numpy(); s = fit["suna"].to_numpy()
    if FIT_SLOPE and np.ptp(b) > 0:
        GAIN, OFFSET = np.polyfit(b, s, 1)
    else:
        GAIN, OFFSET = 1.0, float(np.mean(s - b))
    pred = OFFSET + GAIN * b; ss_res = np.sum((s-pred)**2); ss_tot = np.sum((s-s.mean())**2)
    R2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    resid = s - pred
    print(f"\ncalibration (FIT_SLOPE={FIT_SLOPE}, homogeneous points only):  SUNA = {OFFSET:.4f} + {GAIN:.4f} * bottle")
    print(f"  offset = {OFFSET:.3f} uM | gain = {GAIN:.4f} | R2 = {R2:.4f} | residual std = {resid.std():.3f} uM | n = {len(fit)}")
    print(f"  correction applied to profiles:  nitrate_corr = (SUNA - ({OFFSET:.4f})) / {GAIN:.4f}")
else:
    print("\nnot enough homogeneous pairs to fit; relax HYST_TOL or check the leg split.")
cal.to_csv(OUT_DIR / f"{CRUISE_ID}__suna_bottle_calibration.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ------------------------------------------------------------------ apply correction -> L3
if not np.isnan(OFFSET):
    L3 = L2.copy()
    L3["nitrate_corr_uM"] = (L3["nitrate_uM"] - OFFSET) / GAIN
    L3.attrs["offset"] = OFFSET; L3.attrs["gain"] = GAIN
    keep = L3[(L3["in_cast"]==True) & (L3["L1_clean"]==True) & L3["pressure_db"].notna()]
    keep.to_csv(L3_DIR / f"{CRUISE_ID}__suna_L3_corrected.csv", index=False, encoding="utf-8-sig")
    print(f"wrote L3 corrected profiles: {int(len(keep))} in-cast clean frames -> {L3_DIR}")
    print(f"  surface check (corrected NO3 at <10 db): median {keep.loc[keep['pressure_db']<10,'nitrate_corr_uM'].median():.2f} uM (expect ~0)")
else:
    L3 = L2.copy(); L3["nitrate_corr_uM"] = np.nan
    print("no calibration; L3 not written")

In [ ]:
# ------------------------------------------------------------------ calibration + corrected-profile figures
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
if not np.isnan(OFFSET):
    # (1) calibration scatter
    fig=Figure(figsize=(5.4,5.2)); FigureCanvasAgg(fig); ax=fig.subplots()
    for cst,gp in fit.groupby("cast"):
        ax.plot(gp["bottle"],gp["suna"],"o",ms=7,label=cst)
    xx=np.linspace(min(0,fit["bottle"].min()),fit["bottle"].max()*1.05,50)
    ax.plot(xx,OFFSET+GAIN*xx,"-",color="#333",lw=1.3,label=f"fit: {OFFSET:.2f}+{GAIN:.3f}x")
    ax.plot(xx,xx,"--",color="#aaa",lw=1,label="1:1")
    ax.set_xlabel("bottle NO3 (uM)"); ax.set_ylabel("SUNA raw NO3 (uM)")
    ax.set_title(f"{CRUISE_ID} SUNA-bottle calibration (R2={R2:.3f}, n={len(fit)})",fontsize=10)
    ax.legend(fontsize=8,frameon=False)
    for sp in ("top","right"): ax.spines[sp].set_visible(False)
    fig.tight_layout(); out=OUT_DIR/f"{CRUISE_ID}__suna_calibration_scatter.png"; fig.savefig(out,dpi=135)
    try: plt.close(fig)
    except Exception: pass
    if Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: pass
    print("  saved",out.name)
    # (2) corrected profiles per calibrated cast, bottles overlaid
    for cid in sorted(fit["cast"].unique()):
        d=L3[(L3["cast"]==cid)&(L3["L1_clean"]==True)&L3["pressure_db"].notna()]
        bd=bot[bot["cast_id"]==cid]
        if d.empty: continue
        fig=Figure(figsize=(5.0,6.2)); FigureCanvasAgg(fig); ax=fig.subplots()
        ax.plot(d["nitrate_uM"],d["pressure_db"],".",ms=2.5,color="#c9c9c9",label="raw SUNA")
        ax.plot(d["nitrate_corr_uM"],d["pressure_db"],".",ms=2.5,color="#2E6FB0",label="corrected")
        ax.plot(bd["nitrate_uM"],bd["depth_m"],"s",ms=8,color="#d62728",label="bottle",zorder=5)
        ax.invert_yaxis(); ax.axvline(0,color="#bbb",lw=0.6)
        ax.set_xlabel("NO3 (uM)"); ax.set_ylabel("pressure (db)")
        ax.set_title(f"{CRUISE_ID} {cid} corrected",fontsize=10); ax.legend(fontsize=8,frameon=False)
        for sp in ("top","right"): ax.spines[sp].set_visible(False)
        fig.tight_layout(); out=OUT_DIR/f"{CRUISE_ID}__suna_corrected__{cid}.png"; fig.savefig(out,dpi=130)
        try: plt.close(fig)
        except Exception: pass
        if Image is not None:
            try: display(Image(filename=str(out)))
            except Exception: pass
        print("  saved",out.name)

## 5. Calibration statistics

In [ ]:
# ------------------------------------------------------------------ refactor: calibration as a function
def compute_calibration(depth_win=DEPTH_WIN, fit_slope=FIT_SLOPE, include_flagged=INCLUDE_FLAGGED,
                        min_per_leg=MIN_PER_LEG, hyst_tol=HYST_TOL):
    cal = HC.homogeneous_pairs(L2, bot, depth_win, hyst_tol, min_per_leg, include_flagged, SUNA_VALUE)
    fit = cal[cal["used"]].dropna(subset=["suna"])
    r = HC.fit_line(fit["bottle"], fit["suna"], fit_slope)
    return cal, fit, r["offset"], r["gain"], r["R2"], r["cov"]

_T975={1:12.71,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,8:2.306,9:2.262,10:2.228}
def calibration_stats(fit, off, gn, r2, cov):
    b=fit["bottle"].to_numpy(); s=fit["suna"].to_numpy(); n=len(fit); dof=max(n-2,1)
    corr_at=(s-off)/gn
    rmse_raw=float(np.sqrt(np.mean((s-b)**2)))
    rmse_cor=float(np.sqrt(np.mean((corr_at-b)**2)))
    bias_raw=float(np.mean(s-b)); bias_cor=float(np.mean(corr_at-b))
    r=float(np.corrcoef(b,s)[0,1]) if np.ptp(b)>0 else np.nan
    tcrit=_T975.get(dof,1.96)
    se_gn=se_off=np.nan
    if cov is not None:
        se_gn=float(np.sqrt(cov[0,0])); se_off=float(np.sqrt(cov[1,1]))
    return dict(n=n, offset=off, gain=gn, R2=r2, pearson_r=r,
                se_offset=se_off, se_gain=se_gn, tcrit=tcrit,
                off_lo=off-tcrit*se_off if np.isfinite(se_off) else np.nan,
                off_hi=off+tcrit*se_off if np.isfinite(se_off) else np.nan,
                gain_lo=gn-tcrit*se_gn if np.isfinite(se_gn) else np.nan,
                gain_hi=gn+tcrit*se_gn if np.isfinite(se_gn) else np.nan,
                rmse_raw_uM=rmse_raw, rmse_corrected_uM=rmse_cor,
                bias_raw_uM=bias_raw, bias_corrected_uM=bias_cor)

## 6. Interactive calibration explorer

In [ ]:
# ------------------------------------------------------------------ interactive: calibration explorer
# Tune the depth-matching window and fit options and watch the calibration respond. Left: SUNA-bottle
# scatter with the live fit; right: the selected cast's corrected profile with its bottles.
try:
    import ipywidgets as W
    _HAVE_W = not __import__("os").environ.get("SUNCAT_HEADLESS")   # no widgets in a headless run (tools/run_notebook.py)
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets ipympl then Restart + Run All.")

def _explorer_png(cast_sel, depth_win, fit_slope, include_flagged):
    import io
    cal,fit,off,gn,r2,cov = compute_calibration(depth_win,fit_slope,include_flagged)
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig=Figure(figsize=(12.5,6.0)); FigureCanvasAgg(fig); ax1,ax2=fig.subplots(1,2)
    # left: calibration scatter
    if len(fit)>=2 and np.isfinite(off):
        for cst,gp in fit.groupby("cast"):
            ax1.plot(gp["bottle"],gp["suna"],"o",ms=7,label=cst)
        xx=np.linspace(min(0,fit["bottle"].min()),fit["bottle"].max()*1.05,50)
        ax1.plot(xx,off+gn*xx,"-",color="#333",lw=1.3,label=f"fit {off:.2f}+{gn:.3f}x")
        ax1.plot(xx,xx,"--",color="#aaa",lw=1,label="1:1")
        st=calibration_stats(fit,off,gn,r2,cov)
        ax1.set_title(f"calibration  R2={r2:.3f}  RMSE {st['rmse_raw_uM']:.2f}->{st['rmse_corrected_uM']:.2f} uM  n={len(fit)}",fontsize=9)
    else:
        ax1.text(0.5,0.5,"< 2 usable pairs",ha="center"); 
    ax1.set_xlabel("bottle NO3 (uM)"); ax1.set_ylabel("SUNA raw NO3 (uM)")
    ax1.legend(fontsize=7,frameon=False)
    for sp in ("top","right"): ax1.spines[sp].set_visible(False)
    # right: corrected profile for the chosen cast
    if np.isfinite(off) and cast_sel!="(none)":
        d=L2[(L2["cast"]==cast_sel)&(L2["L1_clean"]==True)&L2["pressure_db"].notna()].copy()
        d["corr"]=(d["nitrate_uM"]-off)/gn
        bd=bot[bot["cast_id"]==cast_sel]
        ax2.plot(d["nitrate_uM"],d["pressure_db"],".",ms=2.3,color="#c9c9c9",label="raw")
        ax2.plot(d["corr"],d["pressure_db"],".",ms=2.3,color="#2E6FB0",label="corrected")
        ax2.plot(bd["nitrate_uM"],bd["depth_m"],"s",ms=8,color="#d62728",label="bottle",zorder=5)
        ax2.invert_yaxis(); ax2.axvline(0,color="#bbb",lw=0.6)
        ax2.set_title(f"{cast_sel} profile (window +/-{depth_win:.1f} db)",fontsize=9)
    ax2.set_xlabel("NO3 (uM)"); ax2.set_ylabel("pressure (db)"); ax2.legend(fontsize=7,frameon=False)
    for sp in ("top","right"): ax2.spines[sp].set_visible(False)
    fig.suptitle(f"{CRUISE_ID} SUNA-bottle calibration explorer",fontsize=11)
    fig.tight_layout(rect=[0,0,1,0.96])
    buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=118); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

if _HAVE_W:
    _casts=sorted(bot["cast_id"].unique().tolist())
    wcast=W.Dropdown(options=_casts+["(none)"],value=_casts[0] if _casts else "(none)",description="cast")
    wwin =W.FloatSlider(min=0.5,max=5.0,step=0.5,value=DEPTH_WIN,description="depth +/- db",
                        continuous_update=False,layout=W.Layout(width="45%"))
    wslope=W.Checkbox(value=FIT_SLOPE,description="fit gain (slope)")
    wflag =W.Checkbox(value=INCLUDE_FLAGGED,description="include flagged bottles")
    def _r(cast_sel,depth_win,fit_slope,include_flagged):
        display(Image(data=_explorer_png(cast_sel,depth_win,fit_slope,include_flagged)))
    out=W.interactive_output(_r,{"cast_sel":wcast,"depth_win":wwin,"fit_slope":wslope,"include_flagged":wflag})
    display(W.VBox([W.HBox([wcast,wwin]),W.HBox([wslope,wflag]),out]))